# Machine Learning Methods for Data Streams
## Project: Concept Drift Detection in Polarized News Streams

In [5]:
import requests
import trafilatura
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000001F420FA98A0> (for post_run_cell), with arguments args (<ExecutionResult object at 1f4212dda30, execution_count=5 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 1f4212de4b0, raw_cell="import requests
import trafilatura
import json
imp.." transformed_cell="import requests
import trafilatura
import json
imp.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/c%3A/Users/krzys/OneDrive/Pulpit/Studia/StudiaDS/2sem/MLDSt/Project/Concept-Drift-Detector/experiments/01_web_scraping.ipynb#W1sZmlsZQ%3D%3D> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

### 1. Web scraping

We used the GDELT database, which provides a comprehensive collection of news articles from around the world. 

The scraper retrieves article URLs based on specified keywords and date ranges (using `GDELT API`), then scrapes the full text of each article (using 'trafilatura' library). The scraped data is stored in 'jsonl' format, which allows for easy processing and analysis.

In [ ]:
SOURCES = {
    "motherjones.com": 0,  #left leaning
    "thenation.com": 0,    #left leaning
    "msnbc.com": 0, 
    "huffpost.com": 0,
    
    "breitbart.com": 1,     #right leaning
    "dailycaller.com" : 1     #right leaning

}


KEYWORDS = "Trump OR Biden OR President"

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2026, 5, 1)

In [ ]:
def get_article_urls_from_gdelt(domain, query, start_date, end_date, max_retries=5):
    """
    Download list of article URLs from GDELT API for a given domain and query within a date range.
    """
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    start_str = start_date.strftime("%Y%m%d%H%M%S")
    end_str = end_date.strftime("%Y%m%d%H%M%S")
    full_query = f'({query}) domain:{domain}'
    
    params = {
        "query": full_query,
        "mode": "artlist",        
        "maxrecords": 20,         
        "sort": "DateDesc",
        "format": "json",
        "startdatetime": start_str,
        "enddatetime": end_str
    }
    
    for attempt in range(max_retries):
        try:
            time.sleep(3) 
            response = requests.get(url, params=params, timeout=30)
            
            if response.status_code == 429:
                wait_time = 5
                print(f"   [API] Error. Trying again. (Attempt {attempt+1}/{max_retries})...")
                time.sleep(wait_time)
                continue
                
            response.raise_for_status()
            data = response.json()
            
            if "articles" in data and len(data["articles"]) > 0:
                return [(article["url"], article["seendate"]) for article in data["articles"]]
            else:
                print(f"   [API] Found 0 articles. Trying again. (Attempt {attempt+1}/{max_retries})...")
                time.sleep(5)
                continue
                
        except requests.exceptions.Timeout:
            print(f"   [API] Timeout. Trying again. (Attempt {attempt+1}/{max_retries})...")
            time.sleep(5)
        except requests.exceptions.ConnectionError:
            print(f"   [API] Connection Error. Trying again. (Attempt {attempt+1}/{max_retries})...")
            time.sleep(5)
            
    print(f"   [API] {max_retries} failed attempts. Returning no articles.")
    return []

In [ ]:
def scrape_article_text(url):
    """
    Download and extract the main text content from a news article URL using trafilatura package.
    """

    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded, include_comments=False, include_tables=False)
    return None

In [ ]:
def save_articles(output_file):
    """
    Main pipeline for saving articles into a JSON file.
    """
    print(f"Begining collection of articles to file {OUTPUT_FILE}")
        
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
        for domain, label in SOURCES.items():
            print(f"Domain: {domain}")
                
            current_date = START_DATE
                
            while current_date < END_DATE:
                print(f"{current_date.strftime('%Y-%m-%d')}...")
                
                next_date = current_date + timedelta(days=1)
                urls_with_dates = get_article_urls_from_gdelt(domain, KEYWORDS, current_date, next_date)
                print(f" Found {len(urls_with_dates)} links.")
                    
                for url, date_str in urls_with_dates:
                    text = scrape_article_text(url)
                        
                    if text and len(text) > 500: 
                        clean_date = datetime.strptime(date_str, "%Y%m%dT%H%M%SZ").isoformat()
        
                        record = {
                            "timestamp": clean_date,
                            "domain": domain,
                            "label": label,
                            "text": text,
                            "url": url
                        }
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        
                    time.sleep(1) 
                
                current_date = next_date

In [ ]:
OUTPUT_FILE = "data/thenation.jsonl" 
save_articles(OUTPUT_FILE)

### Combining multiple JSONL files into one

In [ ]:
def combine_and_deduplicate_jsonl(input_files, output_file):
    seen_values = set()
    total_saved = 0
    
    with open(output_file, 'w', encoding='utf-8') as out_file:
        for input_filename in input_files:
            with open(input_filename, 'r', encoding='utf-8') as in_file:
                for line in in_file:
                    if line.strip():
                        obj = json.loads(line)
                        check_value = obj.get("text", "")
                        
                        # If this value has not occurred before, we save it
                        if check_value not in seen_values:
                            out_file.write(line)
                            seen_values.add(check_value)
                            total_saved += 1
                            
    print(f"Success! {total_saved} unique articles have been saved to '{output_file}'.")

In [ ]:
files_to_combine = ["data/thenation2.jsonl", "data/news_dataset.jsonl"]
output_destination = "data/news_dataset2.jsonl"
    
combine_and_deduplicate_jsonl(files_to_combine, output_destination)

### Audit collected data

In [6]:
def audit_data_stream(file_path):
    """
    Check the monthly distribution of articles by label and visualize it with a bar chart.
    """
    data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))

    df = pd.DataFrame(data)
    df['datetime'] = pd.to_datetime(df['timestamp'])
    df['month'] = df['datetime'].dt.to_period('M')

    monthly_summary = df.groupby(['month', 'label']).size().unstack(fill_value=0)
    monthly_summary = monthly_summary.rename(columns={0: 'Left', 1: 'Right'})
    monthly_summary = monthly_summary.reset_index()
    monthly_summary['month'] = monthly_summary['month'].astype(str)

    totals_row = pd.DataFrame({
        'month': ['Total'],
        'Left': [monthly_summary['Left'].sum()],
        'Right': [monthly_summary['Right'].sum()]
    })
    monthly_summary = pd.concat([monthly_summary, totals_row], ignore_index=True)
    monthly_summary_plot = monthly_summary[monthly_summary['month'] != 'Total']
    
    #print(monthly_summary)

    monthly_summary_plot.set_index('month')[['Left', 'Right']].plot(
        kind='bar', figsize=(14, 6), width=0.8, color=['red', 'blue']
    )
    
    plt.title('Monthly distribution of articles (Left vs Right)', fontsize=14, fontweight='bold')
    plt.xlabel('Month', fontsize=12)
    plt.ylabel('Number of downloaded articles', fontsize=12)
    plt.legend(title='Label', fontsize=10)
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    return monthly_summary

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000001F420FA98A0> (for post_run_cell), with arguments args (<ExecutionResult object at 1f4212e7530, execution_count=6 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 1f4212df260, raw_cell="def audit_data_stream(file_path):
    """
    Chec.." transformed_cell="def audit_data_stream(file_path):
    """
    Chec.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/c%3A/Users/krzys/OneDrive/Pulpit/Studia/StudiaDS/2sem/MLDSt/Project/Concept-Drift-Detector/experiments/01_web_scraping.ipynb#X15sZmlsZQ%3D%3D> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [7]:
combined_file = 'data/news_dataset2.jsonl'
audit_data_stream(combined_file)

AttributeError: module 'matplotlib' has no attribute 'colors'

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000001F420FA98A0> (for post_run_cell), with arguments args (<ExecutionResult object at 1f4216205f0, execution_count=7 error_before_exec=None error_in_exec=module 'matplotlib' has no attribute 'colors' info=<ExecutionInfo object at 1f421620b30, raw_cell="combined_file = 'data/news_dataset2.jsonl'
audit_d.." transformed_cell="combined_file = 'data/news_dataset2.jsonl'
audit_d.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/c%3A/Users/krzys/OneDrive/Pulpit/Studia/StudiaDS/2sem/MLDSt/Project/Concept-Drift-Detector/experiments/01_web_scraping.ipynb#X16sZmlsZQ%3D%3D> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

### Create balanced data stream

In [8]:
from src.preprocess_real_data import extract_and_split_minority_per_day

FILE_PATH = "data/news_dataset2.jsonl" 
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    rows = f.readlines()

data_stream = extract_and_split_minority_per_day(rows)


OUTPUT_PATH = "data/news_dataset2_balanced.jsonl"

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for item in data_stream:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Saved {len(data_stream)} articles to file: {OUTPUT_PATH}")



Total days processed: 829
Days skipped (missing one class): 201
Majority class articles used: 10223
Minority class parts created: 10223
Total articles in stream: 20446
Saved 20446 articles to file: data/news_dataset2_balanced.jsonl
Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000001F420FA98A0> (for post_run_cell), with arguments args (<ExecutionResult object at 1f42ccaaa20, execution_count=8 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 1f42cca8650, raw_cell="from src.preprocess_real_data import extract_and_s.." transformed_cell="from src.preprocess_real_data import extract_and_s.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/c%3A/Users/krzys/OneDrive/Pulpit/Studia/StudiaDS/2sem/MLDSt/Project/Concept-Drift-Detector/experiments/01_web_scraping.ipynb#X21sZmlsZQ%3D%3D> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [9]:
from src.preprocess_real_data import undersample_stream

FILE_PATH = "data/news_dataset2.jsonl" 
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    rows = f.readlines()

data_stream = undersample_stream(rows)


OUTPUT_PATH = "data/news_dataset2_undersampled.jsonl"

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for item in data_stream:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Saved {len(data_stream)} articles to file: {OUTPUT_PATH}")



Total Class 0 (Left) articles before balancing: 2231
Total Class 1 (Right) articles before balancing: 12508
Total Class 0 (Left) articles after balancing: 2231
Total Class 1 (Right) articles after balancing: 2231
Total articles in stream: 4462
Saved 4462 articles to file: data/news_dataset2_undersampled.jsonl
Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000001F420FA98A0> (for post_run_cell), with arguments args (<ExecutionResult object at 1f420fc0320, execution_count=9 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 1f420fc0950, raw_cell="from src.preprocess_real_data import undersample_s.." transformed_cell="from src.preprocess_real_data import undersample_s.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/c%3A/Users/krzys/OneDrive/Pulpit/Studia/StudiaDS/2sem/MLDSt/Project/Concept-Drift-Detector/experiments/01_web_scraping.ipynb#X22sZmlsZQ%3D%3D> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'